# 08 — Backup Name Movers após knockout causal

Reproduzimos a intervenção da seção 3.4 de arXiv:2211.00593 com datasets separados: um target IOI e uma referência pABC balanceada. As três Name Movers primárias são substituídas, somente em `END`, pela média streaming de `hook_result` condicionada ao grupo exato `(pattern, template_id)` e à posição absoluta. Medimos IO−S, escrita direta IO−S e atenção `END→IO`, `END→S1` e `END→S2` nas oito Backup Name Movers publicadas.

In [1]:
import hashlib
import importlib.metadata
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import plotly.express as px
import torch

from ioi_replication.paper import (
    PUBLISHED_CIRCUIT,
    CircuitClass,
    build_balanced_paired_dataset,
    build_paired_dataset,
    cache_grouped_mean_activations,
    measure_backup_name_movers,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED_TARGET = 308
SEED_REFERENCE = 309
N_TARGET = 16
EXAMPLES_PER_REFERENCE_GROUP = 10
N_REFERENCE = 30 * EXAMPLES_PER_REFERENCE_GROUP
BATCH_SIZE_REFERENCE = 16
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/08_backup_name_movers.json"

device = select_device()
model = load_gpt2_small(device)
target_dataset = build_paired_dataset(
    model.tokenizer,
    n=N_TARGET,
    seed=SEED_TARGET,
)
reference_dataset = build_balanced_paired_dataset(
    model.tokenizer,
    examples_per_group=EXAMPLES_PER_REFERENCE_GROUP,
    seed=SEED_REFERENCE,
)
group_means = cache_grouped_mean_activations(
    model,
    reference_dataset.abc_tokens.to(device),
    reference_dataset.groups,
    activation="result",
    batch_size=BATCH_SIZE_REFERENCE,
    min_group_size=EXAMPLES_PER_REFERENCE_GROUP,
    reference_config={
        "kind": "balanced_pABC",
        "seed": SEED_REFERENCE,
        "n": len(reference_dataset),
        "examples_per_group": EXAMPLES_PER_REFERENCE_GROUP,
    },
    reference_sha256=reference_dataset.dataset_sha256,
)
experiment = measure_backup_name_movers(
    model=model,
    dataset=target_dataset,
    group_means=group_means,
)

print(
    f"device={device}; N_target={len(target_dataset)}; "
    f"N_ref={len(reference_dataset)}; grupos_ref={len(group_means.group_counts)}; "
    f"min_grupo={min(group_means.group_counts.values())}"
)
print(
    f"IO−S: {experiment.baseline_logit_difference.mean:+.4f} → "
    f"{experiment.knockout_logit_difference.mean:+.4f} "
    f"(Δ={experiment.delta_logit_difference.mean:+.4f})"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=16; grupos=13
IO−S: +3.1144 → +2.9733 (Δ=-0.1410)


## Contribuição basal, compensação e reescala

**Contribuição basal** é a DLA ou a atenção no forward intacto. A compensação principal é `Δresult/fixed-scale`: a mudança do `hook_result` projetada com a escala basal de `ln_final`. O termo `Δscale` mede separadamente quanto a reescala final altera a DLA; sua soma com `Δresult/fixed-scale` reconstrói `ΔDLA` total.

Cabeças 9.0 e 9.7 estão na mesma camada do primeiro knockout em `hook_result`: a intervenção não pode retroagir para mudar seus resultados ou sua atenção já calculados. Elas permanecem como diagnóstico, mas são excluídas do ranking de compensação.

In [2]:
def summarize(values):
    values = values.detach().float()
    se = values.std(unbiased=True) / values.numel() ** 0.5 if values.numel() > 1 else torch.tensor(0.0)
    return {
        "mean": values.mean().item(),
        "standard_error": se.item(),
        "values": values.cpu().tolist(),
    }

plot_rows = []
attention_rows = []
head_payload = {}
for (layer, head), metrics in experiment.heads.items():
    label = f"{layer}.{head}"
    head_payload[label] = {
        "is_causally_downstream": metrics.is_causally_downstream,
        "baseline_dla": summarize(metrics.baseline_dla.values),
        "knockout_dla": summarize(metrics.knockout_dla.values),
        "delta_dla": summarize(metrics.delta_dla.values),
        "delta_result_fixed_scale": summarize(metrics.delta_result_fixed_scale.values),
        "delta_scale": summarize(metrics.delta_scale.values),
        "baseline_attention_io": summarize(metrics.baseline_attention_io.values),
        "knockout_attention_io": summarize(metrics.knockout_attention_io.values),
        "delta_attention_io": summarize(metrics.delta_attention_io.values),
        "baseline_attention_s1": summarize(metrics.baseline_attention_s1.values),
        "knockout_attention_s1": summarize(metrics.knockout_attention_s1.values),
        "delta_attention_s1": summarize(metrics.delta_attention_s1.values),
        "baseline_attention_s2": summarize(metrics.baseline_attention_s2.values),
        "knockout_attention_s2": summarize(metrics.knockout_attention_s2.values),
        "delta_attention_s2": summarize(metrics.delta_attention_s2.values),
    }
    plot_rows.extend([
        {"head": label, "medida": "DLA basal", "valor": metrics.baseline_dla.mean},
        {"head": label, "medida": "Δresult / escala basal", "valor": metrics.delta_result_fixed_scale.mean},
        {"head": label, "medida": "Δscale", "valor": metrics.delta_scale.mean},
    ])
    attention_rows.extend([
        {"head": label, "posição": "END→IO", "valor": metrics.delta_attention_io.mean},
        {"head": label, "posição": "END→S1", "valor": metrics.delta_attention_s1.mean},
        {"head": label, "posição": "END→S2", "valor": metrics.delta_attention_s2.mean},
    ])

ranking_result = sorted(
    (
        {"head": head, **values["delta_result_fixed_scale"]}
        for head, values in head_payload.items()
        if values["is_causally_downstream"]
    ),
    key=lambda row: row["mean"],
    reverse=True,
)
removed_primary_payload = {
    f"{layer}.{head}": summarize(estimate.values)
    for (layer, head), estimate in experiment.removed_primary_baseline_dla.items()
}
fraction = experiment.compensation_fraction
compensation_fraction_payload = {
    "value": fraction.value,
    "standard_error": fraction.standard_error,
    "ci_95": [fraction.ci_lower, fraction.ci_upper],
    "definition": fraction.definition,
    "warning": fraction.final_logit_delta_warning,
}

print("Ranking causal por Δresult em escala basal:")
for row in ranking_result:
    print(f"  {row['head']}: {row['mean']:+.4f} ± {row['standard_error']:.4f}")
print(
    "Fração de compensação (razão de médias): "
    f"{fraction.value:+.3f} ± {fraction.standard_error:.3f}; "
    f"IC95% [{fraction.ci_lower:+.3f}, {fraction.ci_upper:+.3f}]"
)

Ranking por Δ DLA:
  10.2: +0.5378 ± 0.1173
  10.10: +0.4755 ± 0.0982
  11.2: +0.2097 ± 0.1725
  10.6: +0.1940 ± 0.0289
  10.1: +0.1435 ± 0.0414
  11.9: +0.0852 ± 0.0210
  9.7: +0.0051 ± 0.0017
  9.0: +0.0014 ± 0.0024
Ranking por Δ atenção END→IO menos END→S1:
  10.2: +0.2181 ± 0.0415
  10.10: +0.1949 ± 0.0320
  10.1: +0.1448 ± 0.0305
  10.6: +0.1250 ± 0.0174
  11.9: +0.0777 ± 0.0148
  11.2: +0.0519 ± 0.0272
  9.7: +0.0000 ± 0.0000
  9.0: +0.0000 ± 0.0000


In [3]:
fig_dla = px.bar(
    plot_rows,
    x="head",
    y="valor",
    color="medida",
    barmode="group",
    title="Backup Name Movers: contribuição basal, Δresult e reescala",
    labels={"valor": "projeção IO−S em END", "head": "cabeça"},
)
fig_dla.update_layout(width=980, height=500)
fig_dla.show()

fig_attention = px.bar(
    attention_rows,
    x="head",
    y="valor",
    color="posição",
    barmode="group",
    title="Mudança de atenção após knockout das Name Movers primárias",
    labels={"valor": "Δ atenção", "head": "cabeça"},
)
fig_attention.update_layout(width=980, height=500)
fig_attention.show()

In [4]:
def estimate_payload(estimate):
    return {
        "mean": estimate.mean,
        "standard_error": estimate.standard_error,
        "values": estimate.values.detach().float().cpu().tolist(),
    }

source_paths = {
    "paper/ablation.py": PROJECT_ROOT / "src/ioi_replication/paper/ablation.py",
    "paper/backup.py": PROJECT_ROOT / "src/ioi_replication/paper/backup.py",
    "paper/dataset.py": PROJECT_ROOT / "src/ioi_replication/paper/dataset.py",
    "paper/circuit.py": PROJECT_ROOT / "src/ioi_replication/paper/circuit.py",
    "paper/metrics.py": PROJECT_ROOT / "src/ioi_replication/paper/metrics.py",
}
source_sha256 = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in source_paths.items()
}
source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
git_dirty = bool(
    subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

model_dtype = str(next(model.parameters()).dtype).removeprefix("torch.")
gpu_name = torch.cuda.get_device_name(device) if device.type == "cuda" else None
artifact = {
    "paper": "arXiv:2211.00593",
    "config": {
        "model_name": "gpt2",
        "seed_target": SEED_TARGET,
        "seed_reference": SEED_REFERENCE,
        "n_target": len(target_dataset),
        "n_reference": len(reference_dataset),
        "target_dataset_sha256": target_dataset.dataset_sha256,
        "reference_dataset_sha256": reference_dataset.dataset_sha256,
        "dtype": model_dtype,
        "device": device.type,
        "gpu": gpu_name,
        "reference_batch_size": BATCH_SIZE_REFERENCE,
        "mean_ablation": "pABC exact (pattern, template_id), absolute position, hook_result",
    },
    "reference": {
        "group_counts": {
            f"{pattern}:{template_id}": count
            for (pattern, template_id), count in group_means.group_counts.items()
        },
        "min_group_size": group_means.min_group_size,
        "config": group_means.reference_config,
        "sha256": group_means.reference_sha256,
    },
    "traceability": {
        "source_commit": source_commit,
        "source_sha256": source_sha256,
        "git_dirty": git_dirty,
        "torch_version": torch.__version__,
        "transformer_lens_version": importlib.metadata.version("transformer-lens"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    },
    "logit_difference": {
        "baseline": estimate_payload(experiment.baseline_logit_difference),
        "primary_name_mover_knockout": estimate_payload(experiment.knockout_logit_difference),
        "delta": estimate_payload(experiment.delta_logit_difference),
    },
    "removed_primary_name_movers": removed_primary_payload,
    "backup_heads": head_payload,
    "ranking_by_delta_result_fixed_scale": ranking_result,
    "compensation_fraction": compensation_fraction_payload,
    "interpretation": {
        "baseline": "Contribuição basal no forward intacto.",
        "compensation": "Δresult projetado com a escala basal; exclui heads da camada 9.",
        "rescaling": "Δscale completa a decomposição da DLA total.",
        "attention_positions": ["IO", "S1", "S2"],
        "same_layer": "Heads 9.0 e 9.7 não mudam result/atenção porque o knockout ocorre depois desses cálculos, em hook_result.",
        "fraction_warning": experiment.compensation_fraction.final_logit_delta_warning,
    },
}
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")

Resultado salvo em results\paper\08_backup_name_movers.json


## Interpretação e limites

O ranking causal principal usa apenas `Δresult/fixed-scale` nas seis Backup Name Movers posteriores à camada 9. A DLA total antes/depois permanece como diagnóstico, mas inclui `Δscale`. Em 9.0 e 9.7, resultado e atenção não mudam: o knockout é aplicado em `hook_result` depois que a atenção e os resultados da própria camada 9 já foram calculados; qualquer mudança de DLA total nesses heads é somente reescala final.

A fração de compensação é a razão entre a média da soma de `Δresult/fixed-scale` das backups posteriores e a média da soma da DLA basal das três Name Movers removidas. Ela não é, nem pretende ser, uma identidade com o delta do logit final: caminhos residuais, MLPs e não linearidades permanecem.

O target tem `N_target=16`; a referência pABC tem `N_ref=300`, cobrindo os 30 grupos com 10 exemplos por grupo. As incertezas do efeito no target ainda são amplas, portanto os números são um experimento causal qualitativo reproduzível, não uma estimativa populacional precisa.